# Demo pipeline: Phân loại cảm xúc bình luận (PhoBERT + CNN + Metadata)

Notebook này minh hoạ **từng bước** của pipeline, hữu ích khi làm slide báo cáo / bảo vệ đồ án:

1. Tiền xử lý văn bản (11 bước, từng bước một)
2. Tách từ tiếng Việt (Word Segmentation)
3. Chuẩn hoá metadata
4. Data Augmentation (cân bằng lớp thiểu số)
5. Kiến trúc mô hình
6. Dự đoán trên bình luận mới (cần chạy `python -m src.train` trước)
7. Xem lại kết quả đánh giá (confusion matrix, các chỉ số)

> Chạy notebook này từ thư mục gốc của project (thư mục chứa `config.py`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # chạy từ notebooks/ -> trỏ về thư mục gốc project

import config as cfg
from src.preprocessing.text_cleaner import TextCleaner
from src.preprocessing.word_segmenter import WordSegmenter
from src.preprocessing.metadata_processor import MetadataProcessor
from src.augmentation.eda_augmenter import EDAAugmenter
import pandas as pd

print("Nhãn:", cfg.LABEL_LIST)
print("Cột metadata:", cfg.METADATA_COLUMNS)

## 1. Tiền xử lý văn bản — từng bước một

In [ ]:
cleaner = TextCleaner()

example = "Sản phẩmmmm dc lắmmm ạ!!!! <b>tốt</b> ghê 😊 xem thêm tại https://shop.vn/abc :))"

trace = cleaner.clean_with_trace(example)
for step_name, text_after in trace:
    print(f"{step_name:<28}: {text_after}")

Có thể thấy: HTML bị loại bỏ, emoji/icon chuyển thành từ thể hiện cảm xúc
("tích cực"), URL thay bằng `<url>`, ký tự lặp được nén ("lắmmm" → "lắmm"),
teencode/viết tắt được chuẩn hoá ("dc" → "được").

## 2. Tách từ tiếng Việt (Word Segmentation)

In [ ]:
segmenter = WordSegmenter()
cleaned = cleaner.clean(example)
segmented = segmenter.segment(cleaned)

print("Sau làm sạch :", cleaned)
print("Sau tách từ  :", segmented)
print("\n-> Đây chính là văn bản được đưa vào PhoBERT Tokenizer.")

## 3. Chuẩn hoá metadata

In [ ]:
demo_metadata_df = pd.DataFrame({
    "rating": [5, 3, 1, None],
    "num_likes": [12, 0, 5, 100],
    "verified_purchase": [1, 0, 1, None],
})

mp = MetadataProcessor(schema=cfg.METADATA_SCHEMA, columns=cfg.METADATA_COLUMNS)
transformed = mp.fit_transform(demo_metadata_df)

print("Trước chuẩn hoá:")
display(demo_metadata_df)
print("\nSau chuẩn hoá (rating/num_likes đã scale, verified_purchase giữ nguyên 0/1):")
print(transformed)

## 4. Data Augmentation (EDA) — cân bằng lớp thiểu số

In [ ]:
augmenter = EDAAugmenter(seed=42)
base_text = "sản phẩm rất tốt đóng gói cẩn thận giao hàng nhanh chóng mình rất hài lòng"

print("Gốc:", base_text)
print()
for i, variant in enumerate(augmenter.augment(base_text, num_aug=4), 1):
    print(f"Biến thể {i}: {variant}")

## 5. Kiến trúc mô hình

`PhoBertCnnMetadata` = PhoBERT (contextual embedding) → CNN (nhiều filter
size) → Global Max Pooling **||** Metadata → Dense → **Concatenate → FC →
Dropout → Softmax**.

> Cell dưới đây cần tải `vinai/phobert-base` từ HuggingFace Hub (~500MB,
> cần internet). Nếu chỉ muốn xem cấu trúc code, mở trực tiếp `src/model.py`.

In [ ]:
from src.model import PhoBertCnnMetadata

model = PhoBertCnnMetadata(
    phobert_model_name=cfg.MODEL_CONFIG.phobert_model_name,
    num_classes=cfg.MODEL_CONFIG.num_classes,
    cnn_filter_sizes=cfg.MODEL_CONFIG.cnn_filter_sizes,
    cnn_num_filters=cfg.MODEL_CONFIG.cnn_num_filters,
    metadata_dim=cfg.MODEL_CONFIG.metadata_dim,
    metadata_embed_dim=cfg.MODEL_CONFIG.metadata_embed_dim,
    fc_hidden_dim=cfg.MODEL_CONFIG.fc_hidden_dim,
    dropout=cfg.MODEL_CONFIG.dropout,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"Tổng số tham số: {n_params:,}")
print(model)

## 6. Dự đoán trên bình luận mới

Cell dưới đây cần đã chạy `python -m src.train` trước đó (để có
`outputs/checkpoints/best_model.pt`).

In [ ]:
from src.predict import SentimentPredictor

predictor = SentimentPredictor()

samples = [
    ("Giao hàng nhanh, đóng gói cẩn thận, sản phẩm dùng rất tốt!", {"rating": 5, "num_likes": 12, "verified_purchase": 1}),
    ("Chất lượng kém, không như quảng cáo, thất vọng quá", {"rating": 1, "num_likes": 2, "verified_purchase": 1}),
    ("Bình thường, đúng như mô tả", {"rating": 3, "num_likes": 0, "verified_purchase": 0}),
]

for text, meta in samples:
    result = predictor.predict(text, meta)
    print(f"[{result['label']:>8} | {result['confidence']:.1%}]  {text}")

## 7. Kết quả đánh giá (sau khi chạy `python -m src.evaluate`)

In [ ]:
import json
from IPython.display import Image, display

report_path = os.path.join(cfg.REPORT_DIR, "test_evaluation.json")
cm_path = os.path.join(cfg.REPORT_DIR, "confusion_matrix.png")

if os.path.exists(report_path):
    with open(report_path, encoding="utf-8") as f:
        metrics = json.load(f)
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Macro F1 : {metrics['macro_f1']:.4f}")
    for label, m in metrics["per_class"].items():
        print(f"  {label:>8}: precision={m['precision']:.3f} recall={m['recall']:.3f} f1={m['f1']:.3f}")
else:
    print("Chưa có kết quả đánh giá - hãy chạy `python -m src.evaluate` trước.")

if os.path.exists(cm_path):
    display(Image(filename=cm_path))